In [4]:
import os

DATA_PATH = "/kaggle/input/datasets/isratok/ragdataset11/PH-CRAG_v1.json"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "PH-CRAG_v1.json"

GEN_MODEL_NAME   = "mistralai/Mistral-7B-Instruct-v0.2"
JUDGE_MODEL_NAME = "google/flan-t5-large"
EMBED_MODEL_NAME = "BAAI/bge-large-en-v1.5"
ALPHA = 0.2
POOL  = 50

print(f"Dataset : {DATA_PATH}  exists={os.path.exists(DATA_PATH)}")
print(f"Gen     : {GEN_MODEL_NAME}")
print(f"Judge   : {JUDGE_MODEL_NAME}")
print(f"Embed   : {EMBED_MODEL_NAME}")
print(f"Alpha={ALPHA}  Pool={POOL}")
print("Config OK ✅")


Dataset : /kaggle/input/datasets/isratok/ragdataset11/PH-CRAG_v1.json  exists=True
Gen     : mistralai/Mistral-7B-Instruct-v0.2
Judge   : google/flan-t5-large
Embed   : BAAI/bge-large-en-v1.5
Alpha=0.2  Pool=50
Config OK ✅


In [5]:
# ════════════════════════════════════════════════════════════════
# CELL 1 — INSTALL DEPENDENCIES
# DO NOT pin numpy or scipy — Kaggle pre-installs these
# Only install what Kaggle does NOT have by default
# ════════════════════════════════════════════════════════════════

!pip install -q sentence-transformers faiss-cpu accelerate bitsandbytes

# Verify key packages load correctly
import numpy as np
import torch
import faiss
import sentence_transformers
import transformers
import scipy

print("✅ All packages ready")
print(f"  numpy        : {np.__version__}")
print(f"  torch        : {torch.__version__}")
print(f"  transformers : {transformers.__version__}")
print(f"  sentence-t   : {sentence_transformers.__version__}")
print(f"  scipy        : {scipy.__version__}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.8 MB/s eta 0:00:00:00:0100:01
✅ All packages ready
  numpy        : 2.0.2
  torch        : 2.10.0+cu128
  transformers : 5.0.0
  sentence-t   : 5.2.3
  scipy        : 1.16.3


In [6]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

print(f"Loading dataset from {DATA_PATH} ...")
with open(DATA_PATH) as f:
    raw_data = json.load(f)

corpus, doc_ids, meta = [], [], []
for product in raw_data:
    if product["product_name"] == "QuillBot":
        continue
    for c in product["comments"]:
        corpus.append(c["text"])
        doc_ids.append(c["doc_id"])
        meta.append({
            "doc_id":     c["doc_id"],
            "product":    product["product_name"],
            "subproduct": c["subproduct_name"],
            "sentiment":  c["sentiment"],
        })

print(f"Loaded {len(corpus)} comments from 11 products")
assert len(corpus) == 536, f"Expected 536, got {len(corpus)}"

print(f"Loading {EMBED_MODEL_NAME} ...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

test_dim = embed_model.encode(["test"]).shape[1]
print(f"Embedding dim: {test_dim}")
assert test_dim == 1024, f"Expected 1024, got {test_dim}"

print("Encoding 536 comments ...")
embeddings = embed_model.encode(
    corpus,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True,
)
print(f"Embeddings shape: {embeddings.shape}")

index = faiss.IndexFlatL2(1024)
index.add(embeddings)
print(f"FAISS index: {index.ntotal} vectors ✅")

np.save("embeddings.npy", embeddings)
with open("corpus_meta.json", "w") as f:
    json.dump({"corpus": corpus, "meta": meta, "doc_ids": doc_ids}, f)
print("Saved: embeddings.npy, corpus_meta.json ✅")


Loading dataset from /kaggle/input/datasets/isratok/ragdataset11/PH-CRAG_v1.json ...
Loaded 536 comments from 11 products
Loading BAAI/bge-large-en-v1.5 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding dim: 1024
Encoding 536 comments ...


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Embeddings shape: (536, 1024)
FAISS index: 536 vectors ✅
Saved: embeddings.npy, corpus_meta.json ✅


In [8]:
QUERIES = [
    ("Notion","feature",   "What AI writing and note-taking features does Notion offer?"),
    ("Notion","feature",   "Does Notion support collaborative writing or team editing?"),
    ("Notion","feature",   "What templates and database features does Notion provide?"),
    ("Notion","comparison","How does Notion compare to other productivity tools?"),
    ("Notion","comparison","How does Notion compare to Evernote or Obsidian?"),
    ("Notion","comparison","Is Notion better than traditional note-taking apps?"),
    ("Notion","market",    "What is the overall user reception of Notion?"),
    ("Notion","market",    "What do users say about Notion's pricing and value?"),
    ("Notion","market",    "What are the most common complaints about Notion?"),
    ("Writesonic AI Writer","feature",   "What AI content generation features does Writesonic offer?"),
    ("Writesonic AI Writer","feature",   "Does Writesonic support long-form content generation?"),
    ("Writesonic AI Writer","feature",   "What templates does Writesonic provide for marketers?"),
    ("Writesonic AI Writer","comparison","How does Writesonic compare to Jasper or Copy.ai?"),
    ("Writesonic AI Writer","comparison","Is Writesonic better than other AI writing tools for SEO?"),
    ("Writesonic AI Writer","comparison","How does Writesonic's output quality compare to competitors?"),
    ("Writesonic AI Writer","market",    "What do users say about Writesonic overall?"),
    ("Writesonic AI Writer","market",    "What are users' biggest frustrations with Writesonic?"),
    ("Writesonic AI Writer","market",    "Is Writesonic considered good value for money?"),
    ("Copysmith","feature",   "What copywriting features does Copysmith provide?"),
    ("Copysmith","feature",   "Does Copysmith support eCommerce or product description writing?"),
    ("Copysmith","feature",   "What integrations does Copysmith offer?"),
    ("Copysmith","comparison","How does Copysmith compare to other AI writing tools?"),
    ("Copysmith","comparison","How does Copysmith compare to Jasper for marketing copy?"),
    ("Copysmith","comparison","Is Copysmith better than Copy.ai for eCommerce?"),
    ("Copysmith","market",    "What is the general user sentiment toward Copysmith?"),
    ("Copysmith","market",    "What do users say about Copysmith's output quality?"),
    ("Copysmith","market",    "What are the main limitations users report for Copysmith?"),
    ("compose.ai","feature",   "What autocomplete features does compose.ai offer?"),
    ("compose.ai","feature",   "Does compose.ai work across different browsers or platforms?"),
    ("compose.ai","feature",   "What AI personalization features does compose.ai have?"),
    ("compose.ai","comparison","How does compose.ai compare to other writing assistants?"),
    ("compose.ai","comparison","How does compose.ai compare to Grammarly for autocomplete?"),
    ("compose.ai","comparison","Is compose.ai better than other Chrome extensions for writing?"),
    ("compose.ai","market",    "What do users think about compose.ai overall?"),
    ("compose.ai","market",    "What are users' main complaints about compose.ai?"),
    ("compose.ai","market",    "Is compose.ai considered useful for professional writing?"),
    ("Jasper","feature",   "What AI writing features does Jasper provide?"),
    ("Jasper","feature",   "Does Jasper support brand voice customization?"),
    ("Jasper","feature",   "What long-form writing capabilities does Jasper have?"),
    ("Jasper","comparison","How does Jasper compare to Copy.ai or Writesonic?"),
    ("Jasper","comparison","Is Jasper worth the higher price compared to alternatives?"),
    ("Jasper","comparison","How does Jasper's quality compare to cheaper AI writing tools?"),
    ("Jasper","market",    "What is the overall user sentiment toward Jasper?"),
    ("Jasper","market",    "What do users say about Jasper's pricing?"),
    ("Jasper","market",    "What are the most common Jasper user complaints?"),
    ("Grammarly","feature",   "What grammar and writing features does Grammarly offer?"),
    ("Grammarly","feature",   "Does Grammarly support tone detection and style suggestions?"),
    ("Grammarly","feature",   "What plagiarism detection features does Grammarly have?"),
    ("Grammarly","comparison","How does Grammarly compare to other grammar tools?"),
    ("Grammarly","comparison","How does Grammarly compare to ProWritingAid or Hemingway?"),
    ("Grammarly","comparison","Is Grammarly Premium worth it compared to free alternatives?"),
    ("Grammarly","market",    "What do users say about Grammarly overall?"),
    ("Grammarly","market",    "What are users' biggest complaints about Grammarly?"),
    ("Grammarly","market",    "Do users find Grammarly useful for professional writing?"),
    ("Copy.ai","feature",   "What content generation features does Copy.ai have?"),
    ("Copy.ai","feature",   "Does Copy.ai support social media content creation?"),
    ("Copy.ai","feature",   "What workflow automation features does Copy.ai offer?"),
    ("Copy.ai","comparison","How does Copy.ai differ from Jasper or Writesonic?"),
    ("Copy.ai","comparison","Is Copy.ai better than Jasper for marketing teams?"),
    ("Copy.ai","comparison","How does Copy.ai's free plan compare to competitors?"),
    ("Copy.ai","market",    "Is Copy.ai effective for marketing teams?"),
    ("Copy.ai","market",    "What do users say about Copy.ai's ease of use?"),
    ("Copy.ai","market",    "What are the main limitations of Copy.ai reported by users?"),
    ("Peppertype.ai","feature",   "What AI writing features does Peppertype.ai offer?"),
    ("Peppertype.ai","feature",   "Does Peppertype.ai support multiple content formats?"),
    ("Peppertype.ai","feature",   "What makes Peppertype.ai unique among AI writing tools?"),
    ("Peppertype.ai","comparison","How does Peppertype.ai compare to Copy.ai or Jasper?"),
    ("Peppertype.ai","comparison","Is Peppertype.ai a good alternative to more expensive tools?"),
    ("Peppertype.ai","comparison","How does Peppertype.ai's output quality compare to Jasper?"),
    ("Peppertype.ai","market",    "What do users think about Peppertype.ai overall?"),
    ("Peppertype.ai","market",    "What are users' main concerns about Peppertype.ai?"),
    ("Peppertype.ai","market",    "Is Peppertype.ai considered good value for money?"),
    ("Text Blaze","feature",   "What text expansion features does Text Blaze offer?"),
    ("Text Blaze","feature",   "Does Text Blaze support dynamic templates or variables?"),
    ("Text Blaze","feature",   "What automation features does Text Blaze provide?"),
    ("Text Blaze","comparison","How does Text Blaze compare to other productivity tools?"),
    ("Text Blaze","comparison","How does Text Blaze compare to TextExpander or PhraseExpress?"),
    ("Text Blaze","comparison","Is Text Blaze better than other snippet tools?"),
    ("Text Blaze","market",    "What is the overall reception of Text Blaze?"),
    ("Text Blaze","market",    "What do users say about Text Blaze's ease of use?"),
    ("Text Blaze","market",    "What limitations do users report for Text Blaze?"),
    ("AISEO","feature",   "What SEO writing features does AISEO provide?"),
    ("AISEO","feature",   "Does AISEO support keyword optimization and content scoring?"),
    ("AISEO","feature",   "What AI paraphrasing features does AISEO offer?"),
    ("AISEO","comparison","How does AISEO compare to other SEO tools?"),
    ("AISEO","comparison","How does AISEO compare to Surfer SEO or Frase?"),
    ("AISEO","comparison","Is AISEO a good alternative to more expensive SEO writing tools?"),
    ("AISEO","market",    "What do users say about AISEO overall?"),
    ("AISEO","market",    "What are users' main complaints about AISEO?"),
    ("AISEO","market",    "Is AISEO considered effective for improving search rankings?"),
    ("Hemingway Editor","feature",   "What editing features does Hemingway Editor have?"),
    ("Hemingway Editor","feature",   "Does Hemingway Editor support readability scoring?"),
    ("Hemingway Editor","feature",   "What writing style improvements does Hemingway Editor suggest?"),
    ("Hemingway Editor","comparison","How does Hemingway Editor compare to Grammarly?"),
    ("Hemingway Editor","comparison","Is Hemingway Editor better than Grammarly for style editing?"),
    ("Hemingway Editor","comparison","How does Hemingway Editor compare to ProWritingAid?"),
    ("Hemingway Editor","market",    "What is the overall sentiment toward Hemingway Editor?"),
    ("Hemingway Editor","market",    "What do users say about Hemingway Editor's simplicity?"),
    ("Hemingway Editor","market",    "What are the main limitations of Hemingway Editor?"),
]
assert len(QUERIES) == 99
print(f"Queries: {len(QUERIES)} ✅")
for qt in ["feature","comparison","market"]:
    print(f"  {qt}: {sum(1 for _,t,_ in QUERIES if t==qt)}")


Queries: 99 ✅
  feature: 33
  comparison: 33
  market: 33


In [14]:
RELEVANT = {
    ("Notion","What AI writing and note-taking features does Notion offer?"):["P001_C44","P001_C46","P001_C47","P001_C50","P001_C51","P001_C52","P001_C55","P001_C57","P001_C58","P001_C97","P001_C92","P001_C96","P001_C3","P001_C17","P001_C77","P001_C71"],
    ("Notion","Does Notion support collaborative writing or team editing?"):["P001_C3","P001_C17","P001_C44","P001_C52","P001_C55","P001_C57","P001_C71","P001_C77"],
    ("Notion","What templates and database features does Notion provide?"):["P001_C46","P001_C47","P001_C50","P001_C51","P001_C58","P001_C92","P001_C96","P001_C97"],
    ("Notion","How does Notion compare to other productivity tools?"):["P001_C7","P001_C13","P001_C14","P001_C83","P001_C36","P001_C27","P001_C17"],
    ("Notion","How does Notion compare to Evernote or Obsidian?"):["P001_C7","P001_C13","P001_C14","P001_C27","P001_C36"],
    ("Notion","Is Notion better than traditional note-taking apps?"):["P001_C7","P001_C13","P001_C14","P001_C83","P001_C36"],
    ("Notion","What is the overall user reception of Notion?"):["P001_C6","P001_C8","P001_C11","P001_C16","P001_C37","P001_C33","P001_C66","P001_C72","P001_C95","P001_C79","P001_C59","P001_C69"],
    ("Notion","What do users say about Notion's pricing and value?"):["P001_C6","P001_C8","P001_C16","P001_C33","P001_C37","P001_C59","P001_C66"],
    ("Notion","What are the most common complaints about Notion?"):["P001_C11","P001_C16","P001_C33","P001_C59","P001_C69","P001_C72","P001_C79"],
    ("Writesonic AI Writer","What AI content generation features does Writesonic offer?"):["P002_C3","P002_C5","P002_C12","P002_C21","P002_C24","P002_C27","P002_C28","P002_C31","P002_C33","P002_C35","P002_C42"],
    ("Writesonic AI Writer","Does Writesonic support long-form content generation?"):["P002_C3","P002_C5","P002_C12","P002_C21","P002_C28","P002_C31","P002_C33","P002_C42"],
    ("Writesonic AI Writer","What templates does Writesonic provide for marketers?"):["P002_C24","P002_C27","P002_C28","P002_C31","P002_C35"],
    ("Writesonic AI Writer","How does Writesonic compare to Jasper or Copy.ai?"):["P002_C34","P002_C4","P002_C30","P002_C41"],
    ("Writesonic AI Writer","Is Writesonic better than other AI writing tools for SEO?"):["P002_C4","P002_C30","P002_C34","P002_C41"],
    ("Writesonic AI Writer","How does Writesonic's output quality compare to competitors?"):["P002_C4","P002_C30","P002_C34","P002_C41"],
    ("Writesonic AI Writer","What do users say about Writesonic overall?"):["P002_C1","P002_C2","P002_C7","P002_C10","P002_C16","P002_C22","P002_C36","P002_C37","P002_C39","P002_C43"],
    ("Writesonic AI Writer","What are users' biggest frustrations with Writesonic?"):["P002_C2","P002_C7","P002_C16","P002_C36","P002_C39"],
    ("Writesonic AI Writer","Is Writesonic considered good value for money?"):["P002_C1","P002_C10","P002_C22","P002_C37","P002_C43"],
    ("Copysmith","What copywriting features does Copysmith provide?"):["P003_C4","P003_C7","P003_C8","P003_C14","P003_C29","P003_C31"],
    ("Copysmith","Does Copysmith support eCommerce or product description writing?"):["P003_C4","P003_C7","P003_C8","P003_C14","P003_C29"],
    ("Copysmith","What integrations does Copysmith offer?"):["P003_C7","P003_C8","P003_C14","P003_C31"],
    ("Copysmith","How does Copysmith compare to other AI writing tools?"):["P003_C10","P003_C15","P003_C17","P003_C22"],
    ("Copysmith","How does Copysmith compare to Jasper for marketing copy?"):["P003_C10","P003_C15","P003_C17","P003_C22"],
    ("Copysmith","Is Copysmith better than Copy.ai for eCommerce?"):["P003_C10","P003_C15","P003_C17"],
    ("Copysmith","What is the general user sentiment toward Copysmith?"):["P003_C5","P003_C9","P003_C15","P003_C20","P003_C21","P003_C22","P003_C23","P003_C24","P003_C26","P003_C27"],
    ("Copysmith","What do users say about Copysmith's output quality?"):["P003_C5","P003_C9","P003_C20","P003_C21","P003_C23"],
    ("Copysmith","What are the main limitations users report for Copysmith?"):["P003_C15","P003_C21","P003_C24","P003_C26","P003_C27"],
    ("compose.ai","What autocomplete features does compose.ai offer?"):["P004_C2","P004_C3","P004_C4","P004_C5","P004_C9","P004_C15","P004_C22","P004_C24"],
    ("compose.ai","Does compose.ai work across different browsers or platforms?"):["P004_C2","P004_C3","P004_C4","P004_C9","P004_C15"],
    ("compose.ai","What AI personalization features does compose.ai have?"):["P004_C4","P004_C5","P004_C22","P004_C24"],
    ("compose.ai","How does compose.ai compare to other writing assistants?"):["P004_C9","P004_C20","P004_C6"],
    ("compose.ai","How does compose.ai compare to Grammarly for autocomplete?"):["P004_C6","P004_C9","P004_C20"],
    ("compose.ai","Is compose.ai better than other Chrome extensions for writing?"):["P004_C6","P004_C9","P004_C20"],
    ("compose.ai","What do users think about compose.ai overall?"):["P004_C2","P004_C7","P004_C8","P004_C10","P004_C17","P004_C20","P004_C24"],
    ("compose.ai","What are users' main complaints about compose.ai?"):["P004_C7","P004_C8","P004_C10","P004_C17"],
    ("compose.ai","Is compose.ai considered useful for professional writing?"):["P004_C2","P004_C10","P004_C20","P004_C24"],
    ("Jasper","What AI writing features does Jasper provide?"):["P005_C1","P005_C4","P005_C5","P005_C6","P005_C7","P005_C15","P005_C17","P005_C18","P005_C19","P005_C20","P005_C38"],
    ("Jasper","Does Jasper support brand voice customization?"):["P005_C4","P005_C5","P005_C6","P005_C15","P005_C17","P005_C18","P005_C19","P005_C20"],
    ("Jasper","What long-form writing capabilities does Jasper have?"):["P005_C1","P005_C4","P005_C7","P005_C15","P005_C38"],
    ("Jasper","How does Jasper compare to Copy.ai or Writesonic?"):["P005_C11","P005_C13","P005_C35","P005_C10"],
    ("Jasper","Is Jasper worth the higher price compared to alternatives?"):["P005_C10","P005_C11","P005_C13","P005_C35"],
    ("Jasper","How does Jasper's quality compare to cheaper AI writing tools?"):["P005_C10","P005_C11","P005_C13","P005_C35"],
    ("Jasper","What is the overall user sentiment toward Jasper?"):["P005_C2","P005_C3","P005_C9","P005_C14","P005_C29","P005_C33","P005_C36","P005_C10"],
    ("Jasper","What do users say about Jasper's pricing?"):["P005_C2","P005_C3","P005_C9","P005_C29","P005_C33"],
    ("Jasper","What are the most common Jasper user complaints?"):["P005_C3","P005_C9","P005_C14","P005_C29","P005_C36"],
    ("Grammarly","What grammar and writing features does Grammarly offer?"):["P006_C1","P006_C9","P006_C25","P006_C35","P006_C41","P006_C43","P006_C44","P006_C47","P006_C51","P006_C52"],
    ("Grammarly","Does Grammarly support tone detection and style suggestions?"):["P006_C9","P006_C25","P006_C35","P006_C41","P006_C43","P006_C44","P006_C47"],
    ("Grammarly","What plagiarism detection features does Grammarly have?"):["P006_C1","P006_C25","P006_C41","P006_C51","P006_C52"],
    ("Grammarly","How does Grammarly compare to other grammar tools?"):["P006_C11","P006_C14","P006_C29","P006_C47","P006_C49","P006_C50"],
    ("Grammarly","How does Grammarly compare to ProWritingAid or Hemingway?"):["P006_C11","P006_C14","P006_C29","P006_C49","P006_C50"],
    ("Grammarly","Is Grammarly Premium worth it compared to free alternatives?"):["P006_C11","P006_C14","P006_C47","P006_C49","P006_C50"],
    ("Grammarly","What do users say about Grammarly overall?"):["P006_C1","P006_C4","P006_C21","P006_C24","P006_C26","P006_C37","P006_C38","P006_C46","P006_C48"],
    ("Grammarly","What are users' biggest complaints about Grammarly?"):["P006_C4","P006_C21","P006_C26","P006_C37","P006_C38"],
    ("Grammarly","Do users find Grammarly useful for professional writing?"):["P006_C1","P006_C24","P006_C46","P006_C48"],
    ("Copy.ai","What content generation features does Copy.ai have?"):["P007_C3","P007_C19","P007_C28","P007_C34","P007_C36","P007_C45","P007_C47","P007_C52","P007_C58","P007_C66"],
    ("Copy.ai","Does Copy.ai support social media content creation?"):["P007_C3","P007_C19","P007_C28","P007_C34","P007_C45","P007_C47","P007_C52"],
    ("Copy.ai","What workflow automation features does Copy.ai offer?"):["P007_C28","P007_C34","P007_C36","P007_C47","P007_C58","P007_C66"],
    ("Copy.ai","How does Copy.ai differ from Jasper or Writesonic?"):["P007_C40","P007_C41","P007_C49","P007_C50","P007_C61"],
    ("Copy.ai","Is Copy.ai better than Jasper for marketing teams?"):["P007_C40","P007_C41","P007_C49","P007_C50"],
    ("Copy.ai","How does Copy.ai's free plan compare to competitors?"):["P007_C40","P007_C49","P007_C50","P007_C61"],
    ("Copy.ai","Is Copy.ai effective for marketing teams?"):["P007_C1","P007_C5","P007_C6","P007_C8","P007_C10","P007_C18","P007_C21","P007_C46","P007_C53","P007_C56"],
    ("Copy.ai","What do users say about Copy.ai's ease of use?"):["P007_C1","P007_C5","P007_C6","P007_C8","P007_C18","P007_C21","P007_C46"],
    ("Copy.ai","What are the main limitations of Copy.ai reported by users?"):["P007_C10","P007_C53","P007_C56","P007_C61"],
    ("Peppertype.ai","What AI writing features does Peppertype.ai offer?"):["P008_C1","P008_C2","P008_C3","P008_C8","P008_C9","P008_C15","P008_C16","P008_C17","P008_C18","P008_C19","P008_C31","P008_C32"],
    ("Peppertype.ai","Does Peppertype.ai support multiple content formats?"):["P008_C1","P008_C2","P008_C8","P008_C15","P008_C16","P008_C17","P008_C18","P008_C19"],
    ("Peppertype.ai","What makes Peppertype.ai unique among AI writing tools?"):["P008_C3","P008_C9","P008_C31","P008_C32"],
    ("Peppertype.ai","How does Peppertype.ai compare to Copy.ai or Jasper?"):["P008_C5","P008_C11","P008_C12","P008_C24","P008_C25","P008_C50"],
    ("Peppertype.ai","Is Peppertype.ai a good alternative to more expensive tools?"):["P008_C5","P008_C11","P008_C24","P008_C25","P008_C50"],
    ("Peppertype.ai","How does Peppertype.ai's output quality compare to Jasper?"):["P008_C5","P008_C12","P008_C24","P008_C25"],
    ("Peppertype.ai","What do users think about Peppertype.ai overall?"):["P008_C1","P008_C2","P008_C3","P008_C6","P008_C22","P008_C34","P008_C35","P008_C39","P008_C40","P008_C45"],
    ("Peppertype.ai","What are users' main concerns about Peppertype.ai?"):["P008_C6","P008_C22","P008_C34","P008_C39","P008_C40"],
    ("Peppertype.ai","Is Peppertype.ai considered good value for money?"):["P008_C1","P008_C2","P008_C35","P008_C45"],
    ("Text Blaze","What text expansion features does Text Blaze offer?"):["P009_C1","P009_C2","P009_C3","P009_C7","P009_C8","P009_C13","P009_C15","P009_C16","P009_C32"],
    ("Text Blaze","Does Text Blaze support dynamic templates or variables?"):["P009_C1","P009_C2","P009_C7","P009_C8","P009_C13","P009_C15","P009_C32"],
    ("Text Blaze","What automation features does Text Blaze provide?"):["P009_C3","P009_C7","P009_C8","P009_C15","P009_C16"],
    ("Text Blaze","How does Text Blaze compare to other productivity tools?"):["P009_C10","P009_C20","P009_C29","P009_C39"],
    ("Text Blaze","How does Text Blaze compare to TextExpander or PhraseExpress?"):["P009_C10","P009_C20","P009_C29","P009_C39"],
    ("Text Blaze","Is Text Blaze better than other snippet tools?"):["P009_C10","P009_C20","P009_C29"],
    ("Text Blaze","What is the overall reception of Text Blaze?"):["P009_C1","P009_C13","P009_C16","P009_C22","P009_C25","P009_C18","P009_C31","P009_C34"],
    ("Text Blaze","What do users say about Text Blaze's ease of use?"):["P009_C1","P009_C13","P009_C16","P009_C22","P009_C25"],
    ("Text Blaze","What limitations do users report for Text Blaze?"):["P009_C18","P009_C31","P009_C34"],
    ("AISEO","What SEO writing features does AISEO provide?"):["P010_C2","P010_C17","P010_C18","P010_C23","P010_C24","P010_C25","P010_C28","P010_C31","P010_C32","P010_C16A","P010_C22"],
    ("AISEO","Does AISEO support keyword optimization and content scoring?"):["P010_C17","P010_C18","P010_C23","P010_C24","P010_C25","P010_C28","P010_C31","P010_C32"],
    ("AISEO","What AI paraphrasing features does AISEO offer?"):["P010_C2","P010_C16A","P010_C22","P010_C23","P010_C24"],
    ("AISEO","How does AISEO compare to other SEO tools?"):["P010_C5","P010_C8","P010_C15B","P010_C16B","P010_C33"],
    ("AISEO","How does AISEO compare to Surfer SEO or Frase?"):["P010_C5","P010_C8","P010_C15B","P010_C16B"],
    ("AISEO","Is AISEO a good alternative to more expensive SEO writing tools?"):["P010_C5","P010_C8","P010_C33"],
    ("AISEO","What do users say about AISEO overall?"):["P010_C1","P010_C6","P010_C9","P010_C10","P010_C16A","P010_C19","P010_C21","P010_C26","P010_C29","P010_C33"],
    ("AISEO","What are users' main complaints about AISEO?"):["P010_C6","P010_C9","P010_C19","P010_C21","P010_C26"],
    ("AISEO","Is AISEO considered effective for improving search rankings?"):["P010_C1","P010_C10","P010_C29","P010_C33"],
    ("Hemingway Editor","What editing features does Hemingway Editor have?"):["P011_C5","P011_C6","P011_C15","P011_C16","P011_C19","P011_C20","P011_C22","P011_C24","P011_C35","P011_C40","P011_C43"],
    ("Hemingway Editor","Does Hemingway Editor support readability scoring?"):["P011_C5","P011_C6","P011_C15","P011_C16","P011_C19","P011_C20","P011_C22","P011_C24"],
    ("Hemingway Editor","What writing style improvements does Hemingway Editor suggest?"):["P011_C15","P011_C19","P011_C20","P011_C35","P011_C40","P011_C43"],
    ("Hemingway Editor","How does Hemingway Editor compare to Grammarly?"):["P011_C7","P011_C11","P011_C26","P011_C39"],
    ("Hemingway Editor","Is Hemingway Editor better than Grammarly for style editing?"):["P011_C7","P011_C11","P011_C26","P011_C39"],
    ("Hemingway Editor","How does Hemingway Editor compare to ProWritingAid?"):["P011_C7","P011_C11","P011_C26"],
    ("Hemingway Editor","What is the overall sentiment toward Hemingway Editor?"):["P011_C3","P011_C9","P011_C12","P011_C15","P011_C18","P011_C28","P011_C33","P011_C34","P011_C44","P011_C45"],
    ("Hemingway Editor","What do users say about Hemingway Editor's simplicity?"):["P011_C3","P011_C9","P011_C12","P011_C28","P011_C33"],
    ("Hemingway Editor","What are the main limitations of Hemingway Editor?"):["P011_C15","P011_C18","P011_C34","P011_C44","P011_C45"],
}
total = sum(len(v) for v in RELEVANT.values())
assert len(RELEVANT) == 99
print(f"Relevance labels: {len(RELEVANT)} queries, {total} doc_ids ✅")


Relevance labels: 99 queries, 602 doc_ids ✅


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from collections import Counter

print(f"Loading {GEN_MODEL_NAME} ...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
gen_model.eval()
print("Mistral-7B loaded ✅")

# Reload index
embeddings = np.load("embeddings.npy")
with open("corpus_meta.json") as f:
    saved = json.load(f)
corpus  = saved["corpus"]
meta    = saved["meta"]
doc_ids = saved["doc_ids"]
index   = faiss.IndexFlatL2(1024)
index.add(embeddings)

EVIDENCE_PROMPT = (
    "You are a competitor intelligence analyst. "
    "Your answers must be based strictly on the provided product comments. "
    "Do not add any information not present in the evidence. "
    "If the evidence does not answer the query, say: "
    "'The provided comments do not contain information about this.'"
)

def hf_generate(system_prompt, user_prompt, max_new_tokens=300):
    prompt = f"[INST] {system_prompt}\n\n{user_prompt} [/INST]" if system_prompt else f"[INST] {user_prompt} [/INST]"
    inputs = gen_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        output = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=gen_tokenizer.eos_token_id)
    return gen_tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def encode_query(query):
    return embed_model.encode([query], normalize_embeddings=True)

def run_BL1(query):
    return {"answer": hf_generate(None, query), "doc_ids": [], "texts": []}

def run_BL3(query, k=5):
    D, I = index.search(encode_query(query), k)
    ids  = [doc_ids[i] for i in I[0]]
    txts = [corpus[i]  for i in I[0]]
    mts  = [meta[i]    for i in I[0]]
    ctx  = "\n".join(f"[{m['subproduct']}] {t}" for t,m in zip(txts,mts))
    return {"answer": hf_generate(EVIDENCE_PROMPT, f"Evidence:\n{ctx}\n\nQuery: {query}"), "doc_ids": ids, "texts": txts}

def run_RAGSA(query, k=5, pool=None, alpha=None):
    pool  = pool  or POOL
    alpha = alpha or ALPHA
    D, I  = index.search(encode_query(query), pool)
    cands = [(doc_ids[i], corpus[i], meta[i], D[0][idx]) for idx,i in enumerate(I[0])]
    sent_counts = Counter(m["sentiment"] for _,_,m,_ in cands)
    w = {s: 1.0/(cnt**0.5) for s,cnt in sent_counts.items()}
    scored = sorted([(1.0/(1.0+d)*(1+alpha*w[m["sentiment"]]),did,txt,m) for did,txt,m,d in cands], reverse=True)[:k]
    ids  = [did  for _,did,_,_  in scored]
    txts = [txt  for _,_,txt,_  in scored]
    mts  = [m    for _,_,_,m    in scored]
    ctx  = "\n".join(f"[{m['subproduct']}] {t}" for t,m in zip(txts,mts))
    return {"answer": hf_generate(EVIDENCE_PROMPT, f"Evidence:\n{ctx}\n\nQuery: {query}"), "doc_ids": ids, "texts": txts}

# Quick test — Notion 3 queries
print("\nQuick test (3 queries)...")
for _, _, q in QUERIES[:3]:
    bl3 = run_BL3(q, k=5)
    sa  = run_RAGSA(q, k=5)
    print(f"  re-ranked={'YES ✅' if bl3['doc_ids']!=sa['doc_ids'] else 'NO'} | {q[:50]}")

print("\nRunning all 99 queries × 5 conditions ...")
results = []
for i,(product,qtype,query) in enumerate(QUERIES):
    print(f"[{i+1:02d}/99]  {product}  —  {qtype}")
    row = {"product":product,"type":qtype,"query":query}
    row["BL1"]     = run_BL1(query)
    row["BL3_k5"]  = run_BL3(query, k=5)
    row["BL3_k10"] = run_BL3(query, k=10)
    row["SA_k5"]   = run_RAGSA(query, k=5)
    row["SA_k10"]  = run_RAGSA(query, k=10, pool=50)
    results.append(row)
    with open("experiment_outputs.json","w") as f:
        json.dump(results, f, indent=2)
print(f"Done. Saved {len(results)} results ✅")


Loading mistralai/Mistral-7B-Instruct-v0.2 ...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Mistral-7B loaded ✅

Quick test (3 queries)...
  re-ranked=YES ✅ | What AI writing and note-taking features does Noti
  re-ranked=YES ✅ | Does Notion support collaborative writing or team 
  re-ranked=YES ✅ | What templates and database features does Notion p

Running all 99 queries × 5 conditions ...
[01/99]  Notion  —  feature
[02/99]  Notion  —  feature
[03/99]  Notion  —  feature
[04/99]  Notion  —  comparison
[05/99]  Notion  —  comparison
[06/99]  Notion  —  comparison
[07/99]  Notion  —  market
[08/99]  Notion  —  market
[09/99]  Notion  —  market
[10/99]  Writesonic AI Writer  —  feature
[11/99]  Writesonic AI Writer  —  feature
[12/99]  Writesonic AI Writer  —  feature
[13/99]  Writesonic AI Writer  —  comparison
[14/99]  Writesonic AI Writer  —  comparison
[15/99]  Writesonic AI Writer  —  comparison
[16/99]  Writesonic AI Writer  —  market
[17/99]  Writesonic AI Writer  —  market
[18/99]  Writesonic AI Writer  —  market
[19/99]  Copysmith  —  feature
[20/99]  Copysmith  —  f

In [7]:
import torch
from transformers import T5ForConditionalGeneration, AutoTokenizer

print(f"Loading judge: {JUDGE_MODEL_NAME} ...")
judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_NAME)
judge_model = T5ForConditionalGeneration.from_pretrained(
    JUDGE_MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
judge_model.eval()
print("Flan-T5 judge loaded ✅")

with open("experiment_outputs.json") as f:
    results = json.load(f)

def judge_sentence(sentence, context_texts):
    ctx = " ".join(context_texts) if context_texts else "No evidence provided."
    prompt = f"premise: {ctx} hypothesis: {sentence} Does the premise entail the hypothesis? yes or no"
    inputs = judge_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(judge_model.device)
    with torch.no_grad():
        out = judge_model.generate(**inputs, max_new_tokens=5)
    return judge_tokenizer.decode(out[0], skip_special_tokens=True).strip().lower().startswith("no")

def compute_HR(answer, context_texts):
    sents = [s.strip() for s in answer.split(".") if len(s.strip())>10]
    if not sents: return 0.0
    return sum(1 for s in sents if judge_sentence(s, context_texts)) / len(sents)

# Sanity check
print("=== HR SANITY CHECK ===")
cases = [
    ("Notion has an AI writing assistant.",["Notion has an AI writing assistant feature."],"Grounded"),
    ("Notion costs $5 per month.",["Notion is a great productivity app."],"Hallucinated"),
    ("Copy.ai writes Facebook ads.",[],"Hallucinated"),
]
for sent,ctx,exp in cases:
    got = "Hallucinated" if judge_sentence(sent,ctx) else "Grounded"
    print(f"  {'✅' if got==exp else '❌'} Expected={exp} Got={got}")
print()

hr = {"BL1":[],"BL3_k5":[],"BL3_k10":[],"SA_k5":[],"SA_k10":[]}
for i,row in enumerate(results):
    print(f"HR [{i+1:02d}/{len(results)}]  {row['product']}  —  {row['type']}")
    hr["BL1"].append(    compute_HR(row["BL1"]["answer"],     []))
    hr["BL3_k5"].append( compute_HR(row["BL3_k5"]["answer"],  row["BL3_k5"]["texts"]))
    hr["BL3_k10"].append(compute_HR(row["BL3_k10"]["answer"], row["BL3_k10"]["texts"]))
    hr["SA_k5"].append(  compute_HR(row["SA_k5"]["answer"],   row["SA_k5"]["texts"]))
    hr["SA_k10"].append( compute_HR(row["SA_k10"]["answer"],  row["SA_k10"]["texts"]))
    with open("hr_scores.json","w") as f: json.dump(hr,f)

print(f"\nHR macro-avg:")
print(f"  BL-1      = {np.mean(hr['BL1']):.3f}")
print(f"  BL-3  k=5 = {np.mean(hr['BL3_k5']):.3f}")
print(f"  RAG-SA k=5= {np.mean(hr['SA_k5']):.3f}")
print("Saved hr_scores.json ✅")


Loading judge: google/flan-t5-large ...


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Flan-T5 judge loaded ✅
=== HR SANITY CHECK ===
  ✅ Expected=Grounded Got=Grounded
  ❌ Expected=Hallucinated Got=Grounded
  ✅ Expected=Hallucinated Got=Hallucinated

HR [01/99]  Notion  —  feature
HR [02/99]  Notion  —  feature
HR [03/99]  Notion  —  feature
HR [04/99]  Notion  —  comparison
HR [05/99]  Notion  —  comparison
HR [06/99]  Notion  —  comparison
HR [07/99]  Notion  —  market
HR [08/99]  Notion  —  market
HR [09/99]  Notion  —  market
HR [10/99]  Writesonic AI Writer  —  feature
HR [11/99]  Writesonic AI Writer  —  feature
HR [12/99]  Writesonic AI Writer  —  feature
HR [13/99]  Writesonic AI Writer  —  comparison
HR [14/99]  Writesonic AI Writer  —  comparison
HR [15/99]  Writesonic AI Writer  —  comparison
HR [16/99]  Writesonic AI Writer  —  market
HR [17/99]  Writesonic AI Writer  —  market
HR [18/99]  Writesonic AI Writer  —  market
HR [19/99]  Copysmith  —  feature
HR [20/99]  Copysmith  —  feature
HR [21/99]  Copysmith  —  feature
HR [22/99]  Copysmith  —  comparison


In [8]:
from transformers import pipeline as hf_pipeline, AutoTokenizer

NLI_MODEL = "cross-encoder/nli-deberta-v3-large"
print(f"Loading {NLI_MODEL} ...")
nli     = hf_pipeline("text-classification", model=NLI_MODEL, truncation=True, max_length=512)
nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL)
print("DeBERTa loaded ✅")

_preds = nli("The sky is blue. [SEP] The sky is blue.", top_k=None)
entail_label = max(_preds, key=lambda x: x["score"])["label"]
print(f"Entailment label: '{entail_label}'")

with open("experiment_outputs.json") as f:
    results = json.load(f)

def _trunc(text, max_tok):
    toks = nli_tok.encode(text, add_special_tokens=False)
    return nli_tok.decode(toks[:max_tok], skip_special_tokens=True)

def compute_FC(answer, context_texts):
    if not context_texts: return 0.0
    ctx   = _trunc(" ".join(context_texts), 300)
    sents = [s.strip() for s in answer.split(".") if len(s.strip())>10]
    if not sents: return 0.0
    scores = []
    for s in sents:
        preds = nli(ctx + " [SEP] " + _trunc(s,100), top_k=None)
        scores.append(next((p["score"] for p in preds if p["label"]==entail_label), 0.0))
    return float(np.mean(scores))

fc_hi = compute_FC("The sky is blue.",["The sky appears blue."])
fc_lo = compute_FC("The sky is green.",["The sky appears blue."])
print(f"FC sanity: match={fc_hi:.3f} contra={fc_lo:.3f} {'✅' if fc_hi>fc_lo else '❌'}")
print()

fc = {"BL1":[],"BL3_k5":[],"SA_k5":[]}
for i,row in enumerate(results):
    print(f"FC [{i+1:02d}/{len(results)}]  {row['product']}  —  {row['type']}")
    fc["BL1"].append(   compute_FC(row["BL1"]["answer"],    []))
    fc["BL3_k5"].append(compute_FC(row["BL3_k5"]["answer"], row["BL3_k5"]["texts"]))
    fc["SA_k5"].append( compute_FC(row["SA_k5"]["answer"],  row["SA_k5"]["texts"]))
    with open("fc_scores.json","w") as f: json.dump(fc,f)

print(f"\nFC macro-avg:")
print(f"  BL-1     = {np.mean(fc['BL1']):.3f}")
print(f"  BL-3 k=5 = {np.mean(fc['BL3_k5']):.3f}")
print(f"  SA   k=5 = {np.mean(fc['SA_k5']):.3f}")
print("Saved fc_scores.json ✅")


Loading cross-encoder/nli-deberta-v3-large ...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

DeBERTa loaded ✅
Entailment label: 'entailment'
FC sanity: match=0.954 contra=0.000 ✅

FC [01/99]  Notion  —  feature


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


FC [02/99]  Notion  —  feature
FC [03/99]  Notion  —  feature
FC [04/99]  Notion  —  comparison
FC [05/99]  Notion  —  comparison
FC [06/99]  Notion  —  comparison
FC [07/99]  Notion  —  market
FC [08/99]  Notion  —  market
FC [09/99]  Notion  —  market
FC [10/99]  Writesonic AI Writer  —  feature
FC [11/99]  Writesonic AI Writer  —  feature
FC [12/99]  Writesonic AI Writer  —  feature
FC [13/99]  Writesonic AI Writer  —  comparison
FC [14/99]  Writesonic AI Writer  —  comparison
FC [15/99]  Writesonic AI Writer  —  comparison
FC [16/99]  Writesonic AI Writer  —  market
FC [17/99]  Writesonic AI Writer  —  market
FC [18/99]  Writesonic AI Writer  —  market
FC [19/99]  Copysmith  —  feature
FC [20/99]  Copysmith  —  feature
FC [21/99]  Copysmith  —  feature
FC [22/99]  Copysmith  —  comparison
FC [23/99]  Copysmith  —  comparison
FC [24/99]  Copysmith  —  comparison
FC [25/99]  Copysmith  —  market
FC [26/99]  Copysmith  —  market
FC [27/99]  Copysmith  —  market
FC [28/99]  compose.ai 

In [9]:
with open("experiment_outputs.json") as f:
    results = json.load(f)

def p_at_k(ret,rel,k): return sum(1 for d in ret[:k] if d in rel)/k
def r_at_k(ret,rel,k): return sum(1 for d in ret[:k] if d in rel)/max(len(rel),1)

p5s,r5s,p10s,r10s = [],[],[],[]
p5b,r5b,p10b,r10b = [],[],[],[]

for row,(prod,_,query) in zip(results,QUERIES):
    rel = RELEVANT.get((prod,query),[])
    p5s.append( p_at_k(row["SA_k5"]["doc_ids"],  rel,5))
    r5s.append( r_at_k(row["SA_k5"]["doc_ids"],  rel,5))
    p10s.append(p_at_k(row["SA_k10"]["doc_ids"], rel,10))
    r10s.append(r_at_k(row["SA_k10"]["doc_ids"], rel,10))
    p5b.append( p_at_k(row["BL3_k5"]["doc_ids"], rel,5))
    r5b.append( r_at_k(row["BL3_k5"]["doc_ids"], rel,5))
    p10b.append(p_at_k(row["BL3_k10"]["doc_ids"],rel,10))
    r10b.append(r_at_k(row["BL3_k10"]["doc_ids"],rel,10))

ret = {
    "SA":  {"p5":p5s,"r5":r5s,"p10":p10s,"r10":r10s},
    "BL3": {"p5":p5b,"r5":r5b,"p10":p10b,"r10":r10b},
}
with open("retrieval_scores.json","w") as f: json.dump(ret,f)
print(f"RAG-SA  P@5={np.mean(p5s):.3f} R@5={np.mean(r5s):.3f} P@10={np.mean(p10s):.3f} R@10={np.mean(r10s):.3f}")
print(f"BL-3    P@5={np.mean(p5b):.3f} R@5={np.mean(r5b):.3f} P@10={np.mean(p10b):.3f} R@10={np.mean(r10b):.3f}")
print("Saved retrieval_scores.json ✅")


RAG-SA  P@5=0.101 R@5=0.085 P@10=0.079 R@10=0.128
BL-3    P@5=0.109 R@5=0.089 P@10=0.082 R@10=0.133
Saved retrieval_scores.json ✅


In [10]:
from collections import defaultdict

with open("experiment_outputs.json") as f: results = json.load(f)
with open("hr_scores.json")          as f: hr      = json.load(f)
with open("corpus_meta.json")        as f: saved   = json.load(f)

id2sent = {m["doc_id"]:m["sentiment"] for m in saved["meta"]}

def dom_sent(ids):
    if not ids: return "neutral"
    return Counter(id2sent.get(d,"neutral") for d in ids).most_common(1)[0][0]

def compute_SBS(key, hr_list):
    pos,neg = [],[]
    for i,row in enumerate(results):
        d = dom_sent(row[key]["doc_ids"])
        if d=="positive": pos.append(hr_list[i])
        elif d=="negative": neg.append(hr_list[i])
    hp = np.mean(pos) if pos else 0.0
    hn = np.mean(neg) if neg else 0.0
    print(f"  pos_dom={len(pos)} HR={hp:.3f} | neg_dom={len(neg)} HR={hn:.3f}")
    return float(round(hp-hn,3))

prod_sents = defaultdict(list)
for m in saved["meta"]: prod_sents[m["product"]].append(m["sentiment"])
prod_dom = {p:Counter(s).most_common(1)[0][0] for p,s in prod_sents.items()}
print("Product dominant sentiments:")
for p,s in prod_dom.items(): print(f"  {p:<28} {s}")

pos_bl1,neg_bl1 = [],[]
for i,(prod,_,_) in enumerate(QUERIES):
    d = prod_dom.get(prod,"neutral")
    if d=="positive": pos_bl1.append(hr["BL1"][i])
    elif d=="negative": neg_bl1.append(hr["BL1"][i])
SBS_BL1 = float(round((np.mean(pos_bl1) if pos_bl1 else 0.0)-(np.mean(neg_bl1) if neg_bl1 else 0.0),3))
print(f"\nBL-1 SBS = {SBS_BL1:.3f}")

print("BL-3 k=5:");  SBS_BL3_k5  = compute_SBS("BL3_k5",  hr["BL3_k5"])
print("BL-3 k=10:"); SBS_BL3_k10 = compute_SBS("BL3_k10", hr["BL3_k10"])
print("SA k=5:");    SBS_SA_k5   = compute_SBS("SA_k5",   hr["SA_k5"])
print("SA k=10:");   SBS_SA_k10  = compute_SBS("SA_k10",  hr["SA_k10"])

sbs = {"BL1":SBS_BL1,"BL3_k5":SBS_BL3_k5,"BL3_k10":SBS_BL3_k10,"SA_k5":SBS_SA_k5,"SA_k10":SBS_SA_k10}
with open("sbs_scores.json","w") as f: json.dump(sbs,f)
print(f"\nSBS: BL-1={SBS_BL1:.3f}  BL-3={SBS_BL3_k5:.3f}  RAG-SA={SBS_SA_k5:.3f}")
print("Saved sbs_scores.json ✅")


Product dominant sentiments:
  Notion                       neutral
  Writesonic AI Writer         neutral
  Copysmith                    neutral
  compose.ai                   neutral
  Jasper                       neutral
  Grammarly                    neutral
  Copy.ai                      neutral
  Peppertype.ai                neutral
  Text Blaze                   neutral
  AISEO                        neutral
  Hemingway Editor             neutral

BL-1 SBS = 0.000
BL-3 k=5:
  pos_dom=12 HR=0.602 | neg_dom=0 HR=0.000
BL-3 k=10:
  pos_dom=15 HR=0.467 | neg_dom=0 HR=0.000
SA k=5:
  pos_dom=15 HR=0.433 | neg_dom=17 HR=0.576
SA k=10:
  pos_dom=21 HR=0.407 | neg_dom=6 HR=0.597

SBS: BL-1=0.000  BL-3=0.602  RAG-SA=-0.143
Saved sbs_scores.json ✅


In [11]:
with open("hr_scores.json")   as f: hr  = json.load(f)
with open("fc_scores.json")   as f: fc  = json.load(f)
with open("sbs_scores.json")  as f: sbs = json.load(f)
with open("corpus_meta.json") as f: saved = json.load(f)

id2sent = {m["doc_id"]:m["sentiment"] for m in saved["meta"]}
GENERIC_PROMPT = "You are a helpful assistant."

def run_RAGSA_no_prompt(query, k=5):
    D,I = index.search(encode_query(query), POOL)
    cands = [(doc_ids[i],corpus[i],meta[i],D[0][idx]) for idx,i in enumerate(I[0])]
    sent_counts = Counter(m["sentiment"] for _,_,m,_ in cands)
    w = {s:1.0/(cnt**0.5) for s,cnt in sent_counts.items()}
    scored = sorted([(1.0/(1.0+d)*(1+ALPHA*w[m["sentiment"]]),did,txt,m) for did,txt,m,d in cands],reverse=True)[:k]
    ids  = [did  for _,did,_,_ in scored]
    txts = [txt  for _,_,txt,_ in scored]
    mts  = [m    for _,_,_,m   in scored]
    ctx  = "\n".join(f"[{m['subproduct']}] {t}" for t,m in zip(txts,mts))
    return {"answer":hf_generate(GENERIC_PROMPT,f"Evidence:\n{ctx}\n\nQuery: {query}"),"texts":txts,"doc_ids":ids}

print(f"Running ablation ({len(QUERIES)} queries)...")
abl = []
for i,(prod,qt,query) in enumerate(QUERIES):
    print(f"  [{i+1:02d}/{len(QUERIES)}] {prod} — {qt}")
    abl.append(run_RAGSA_no_prompt(query))

hr_np = [compute_HR(r["answer"],r["texts"]) for r in abl]
fc_np = [compute_FC(r["answer"],r["texts"]) for r in abl]

def sbs_from_lists(hr_list,id_lists):
    pos,neg = [],[]
    for i,ids in enumerate(id_lists):
        d = Counter(id2sent.get(x,"neutral") for x in ids).most_common(1)[0][0] if ids else "neutral"
        if d=="positive": pos.append(hr_list[i])
        elif d=="negative": neg.append(hr_list[i])
    return float(round((np.mean(pos) if pos else 0.0)-(np.mean(neg) if neg else 0.0),3))

sbs_np = sbs_from_lists(hr_np,[r["doc_ids"] for r in abl])

abl_hr  = {"SA_full":hr["SA_k5"],  "no_rerank":hr["BL3_k5"],  "no_prompt":hr_np,  "no_ret":hr["BL1"]}
abl_fc  = {"SA_full":fc["SA_k5"],  "no_rerank":fc["BL3_k5"],  "no_prompt":fc_np,  "no_ret":fc["BL1"]}
abl_sbs = {"SA_full":sbs["SA_k5"],"no_rerank":sbs["BL3_k5"],"no_prompt":sbs_np,"no_ret":sbs["BL1"]}

with open("ablation_hr.json","w")  as f: json.dump(abl_hr,f)
with open("ablation_fc.json","w")  as f: json.dump(abl_fc,f)
with open("ablation_sbs.json","w") as f: json.dump(abl_sbs,f)

print(f"\nAblation HR:")
print(f"  RAG-SA full          = {np.mean(abl_hr['SA_full']):.3f}")
print(f"  w/o sentiment rerank = {np.mean(abl_hr['no_rerank']):.3f}")
print(f"  w/o evidence prompt  = {np.mean(abl_hr['no_prompt']):.3f}")
print(f"  w/o retrieval        = {np.mean(abl_hr['no_ret']):.3f}")
print("Saved ablation files ✅")


Running ablation (99 queries)...
  [01/99] Notion — feature
  [02/99] Notion — feature
  [03/99] Notion — feature
  [04/99] Notion — comparison
  [05/99] Notion — comparison
  [06/99] Notion — comparison
  [07/99] Notion — market
  [08/99] Notion — market
  [09/99] Notion — market
  [10/99] Writesonic AI Writer — feature
  [11/99] Writesonic AI Writer — feature
  [12/99] Writesonic AI Writer — feature
  [13/99] Writesonic AI Writer — comparison
  [14/99] Writesonic AI Writer — comparison
  [15/99] Writesonic AI Writer — comparison
  [16/99] Writesonic AI Writer — market
  [17/99] Writesonic AI Writer — market
  [18/99] Writesonic AI Writer — market
  [19/99] Copysmith — feature
  [20/99] Copysmith — feature
  [21/99] Copysmith — feature
  [22/99] Copysmith — comparison
  [23/99] Copysmith — comparison
  [24/99] Copysmith — comparison
  [25/99] Copysmith — market
  [26/99] Copysmith — market
  [27/99] Copysmith — market
  [28/99] compose.ai — feature
  [29/99] compose.ai — feature
  [30

In [12]:
from collections import defaultdict
from scipy import stats

with open("hr_scores.json")        as f: hr  = json.load(f)
with open("fc_scores.json")        as f: fc  = json.load(f)
with open("retrieval_scores.json") as f: ret = json.load(f)
with open("sbs_scores.json")       as f: sbs = json.load(f)
with open("ablation_hr.json")      as f: abl = json.load(f)

S   = lambda lst: f"{np.mean(lst):.3f}"
dhr = lambda a,b: f"{(a-b)/b*100:+.1f}%" if b>0 else "—"

print("="*80)
print("TABLE 4  —  MAIN RESULTS  (macro-average, k=5)")
print("="*80)
print(f"{'System':<24}  {'HR↓':>6}  {'FC↑':>6}  {'P@5↑':>6}  {'R@5↑':>6}  {'P@10↑':>6}  {'R@10↑':>6}  {'SBS↓':>6}")
print("-"*80)
print(f"{'BL-1 (No-RAG)':<24}  {S(hr['BL1']):>6}  {S(fc['BL1']):>6}  {'—':>6}  {'—':>6}  {'—':>6}  {'—':>6}  {sbs['BL1']:>6.3f}")
print(f"{'BL-3 (RAG-Std)':<24}  {S(hr['BL3_k5']):>6}  {S(fc['BL3_k5']):>6}  {S(ret['BL3']['p5']):>6}  {S(ret['BL3']['r5']):>6}  {'—':>6}  {'—':>6}  {sbs['BL3_k5']:>6.3f}")
print(f"{'RAG-SA (proposed)':<24}  {S(hr['SA_k5']):>6}  {S(fc['SA_k5']):>6}  {S(ret['SA']['p5']):>6}  {S(ret['SA']['r5']):>6}  {S(ret['SA']['p10']):>6}  {S(ret['SA']['r10']):>6}  {sbs['SA_k5']:>6.3f}")

print("\n"+"="*80)
print("TABLE 5  —  PER-PRODUCT RESULTS  (RAG-SA, k=5)")
print("="*80)
ph1=defaultdict(list); psa=defaultdict(list); pfc=defaultdict(list); pp5=defaultdict(list)
for i,(prod,_,_) in enumerate(QUERIES):
    ph1[prod].append(hr["BL1"][i]); psa[prod].append(hr["SA_k5"][i])
    pfc[prod].append(fc["SA_k5"][i]); pp5[prod].append(ret["SA"]["p5"][i])
n_map={"Notion":99,"Copy.ai":68,"Grammarly":52,"Peppertype.ai":50,"Hemingway Editor":46,
       "Writesonic AI Writer":43,"Text Blaze":43,"Jasper":40,"AISEO":37,"Copysmith":33,"compose.ai":25}
order=["Notion","Copy.ai","Grammarly","Peppertype.ai","Hemingway Editor",
       "Writesonic AI Writer","Text Blaze","Jasper","AISEO","Copysmith","compose.ai"]
print(f"{'Product':<24}  {'n':>4}  {'HR(BL1)':>8}  {'HR(SA)':>7}  {'Reduc%':>7}  {'FC':>6}  {'P@5':>6}")
print("-"*80)
for p in order:
    b1=np.mean(ph1[p]); sa=np.mean(psa[p])
    r=(b1-sa)/b1*100 if b1>0 else 0.0
    print(f"{p:<24}  {n_map[p]:>4}  {b1:>8.3f}  {sa:>7.3f}  {r:>6.1f}%  {np.mean(pfc[p]):>6.3f}  {np.mean(pp5[p]):>6.3f}")
print("-"*80)
b1a=np.mean(hr['BL1']); saa=np.mean(hr['SA_k5'])
print(f"{'Macro-avg':<24}  {'—':>4}  {b1a:>8.3f}  {saa:>7.3f}  {(b1a-saa)/b1a*100:>6.1f}%  {np.mean(fc['SA_k5']):>6.3f}  {np.mean(ret['SA']['p5']):>6.3f}")

print("\n"+"="*72)
print("TABLE 6  —  RETRIEVAL DEPTH  (k=5 vs k=10)")
print("="*72)
print(f"{'System':<22}  {'k':>3}  {'HR':>6}  {'P@k':>6}  {'R@k':>6}  {'FC':>6}  {'SBS':>6}")
print("-"*60)
print(f"{'BL-3':<22}  {'5':>3}  {S(hr['BL3_k5']):>6}  {S(ret['BL3']['p5']):>6}  {S(ret['BL3']['r5']):>6}  {S(fc['BL3_k5']):>6}  {sbs['BL3_k5']:>6.3f}")
print(f"{'BL-3':<22}  {'10':>3}  {S(hr['BL3_k10']):>6}  {S(ret['BL3']['p10']):>6}  {S(ret['BL3']['r10']):>6}  {'—':>6}  {sbs['BL3_k10']:>6.3f}")
print(f"{'RAG-SA':<22}  {'5':>3}  {S(hr['SA_k5']):>6}  {S(ret['SA']['p5']):>6}  {S(ret['SA']['r5']):>6}  {S(fc['SA_k5']):>6}  {sbs['SA_k5']:>6.3f}")
print(f"{'RAG-SA':<22}  {'10':>3}  {S(hr['SA_k10']):>6}  {S(ret['SA']['p10']):>6}  {S(ret['SA']['r10']):>6}  {'—':>6}  {sbs['SA_k10']:>6.3f}")

print("\n"+"="*55)
print("TABLE 7  —  ABLATION STUDY  (k=5)")
print("="*55)
HRf=np.mean(abl["SA_full"]); HRnr=np.mean(abl["no_rerank"])
HRnp=np.mean(abl["no_prompt"]); HRno=np.mean(abl["no_ret"])
print(f"{'Configuration':<30}  {'HR':>6}  {'ΔHR':>10}")
print("-"*50)
print(f"{'RAG-SA (full)':<30}  {HRf:.3f}  {'—':>10}")
print(f"{'w/o sentiment re-ranking':<30}  {HRnr:.3f}  {dhr(HRnr,HRf):>10}")
print(f"{'w/o evidence-only prompt':<30}  {HRnp:.3f}  {dhr(HRnp,HRf):>10}")
print(f"{'w/o retrieval':<30}  {HRno:.3f}  {dhr(HRno,HRf):>10}")

print("\n"+"="*48)
print("TABLE 8  —  QUERY TYPE BREAKDOWN  (RAG-SA, k=5)")
print("="*48)
thr=defaultdict(list); tfc=defaultdict(list); tp5=defaultdict(list)
for i,(_,qt,_) in enumerate(QUERIES):
    thr[qt].append(hr["SA_k5"][i]); tfc[qt].append(fc["SA_k5"][i]); tp5[qt].append(ret["SA"]["p5"][i])
print(f"{'Query Type':<18}  {'HR':>6}  {'FC':>6}  {'P@5':>6}")
print("-"*40)
for qt in ["feature","comparison","market"]:
    print(f"{qt:<18}  {np.mean(thr[qt]):>6.3f}  {np.mean(tfc[qt]):>6.3f}  {np.mean(tp5[qt]):>6.3f}")
print("-"*40)
print(f"{'Macro-avg':<18}  {np.mean(hr['SA_k5']):>6.3f}  {np.mean(fc['SA_k5']):>6.3f}  {np.mean(ret['SA']['p5']):>6.3f}")

print("\n"+"="*62)
print("SIGNIFICANCE TESTS  (paired t-test, α=0.05)")
print("="*62)
bl1a=np.array(hr["BL1"]); bl3a=np.array(hr["BL3_k5"]); saa=np.array(hr["SA_k5"])
t1,p1=stats.ttest_rel(bl1a,saa)
t2,p2=stats.ttest_rel(bl3a,saa)
t3,p3=stats.ttest_rel(saa,np.array(hr["SA_k10"]))
sig=lambda p: "p<0.05 SIGNIFICANT ✅" if p<0.05 else "NOT significant ❌"
print(f"RAG-SA vs BL-1:     t={t1:+.3f}  p={p1:.4f}  {sig(p1)}")
print(f"RAG-SA vs BL-3:     t={t2:+.3f}  p={p2:.4f}  {sig(p2)}")
print(f"SA k=5  vs SA k=10: t={t3:+.3f}  p={p3:.4f}  {sig(p3)}")
print("\nDONE ✅ — copy tables above into your paper.")


TABLE 4  —  MAIN RESULTS  (macro-average, k=5)
System                       HR↓     FC↑    P@5↑    R@5↑   P@10↑   R@10↑    SBS↓
--------------------------------------------------------------------------------
BL-1 (No-RAG)              0.983   0.000       —       —       —       —   0.000
BL-3 (RAG-Std)             0.608   0.177   0.109   0.089       —       —   0.602
RAG-SA (proposed)          0.501   0.189   0.101   0.085   0.079   0.128  -0.143

TABLE 5  —  PER-PRODUCT RESULTS  (RAG-SA, k=5)
Product                      n   HR(BL1)   HR(SA)   Reduc%      FC     P@5
--------------------------------------------------------------------------------
Notion                      99     1.000    0.398    60.2%   0.140   0.156
Copy.ai                     68     0.979    0.548    44.0%   0.181   0.133
Grammarly                   52     1.000    0.269    73.1%   0.061   0.133
Peppertype.ai               50     0.977    0.502    48.6%   0.262   0.133
Hemingway Editor            46     0.980    

In [1]:
!pip install -q faiss-cpu sentence-transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.1 MB/s eta 0:00:00:00:0100:01


In [2]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

DATA_PATH = "/kaggle/input/datasets/isratok/ragdataset11/PH-CRAG_v1.json"
EMBED_MODEL_NAME = "BAAI/bge-large-en-v1.5"

# Load dataset
with open(DATA_PATH) as f:
    raw_data = json.load(f)

corpus, doc_ids, meta = [], [], []
for product in raw_data:
    if product["product_name"] == "QuillBot":
        continue
    for c in product["comments"]:
        corpus.append(c["text"])
        doc_ids.append(c["doc_id"])
        meta.append({
            "doc_id": c["doc_id"],
            "product": product["product_name"],
            "subproduct": c["subproduct_name"],
            "sentiment": c["sentiment"],
        })

print(f"Loaded {len(corpus)} comments")

# Encode embeddings
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
embeddings = embed_model.encode(corpus, convert_to_numpy=True, normalize_embeddings=True)

# Build FAISS index
index = faiss.IndexFlatL2(1024)
index.add(embeddings)
print(f"FAISS index has {index.ntotal} vectors")

# Save files
np.save("embeddings.npy", embeddings)
with open("corpus_meta.json", "w") as f:
    json.dump({"corpus": corpus, "meta": meta, "doc_ids": doc_ids}, f)
print("✅ Saved embeddings.npy and corpus_meta.json")

Loaded 536 comments


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

FAISS index has 536 vectors
✅ Saved embeddings.npy and corpus_meta.json


In [4]:
# ============================================================
# STANDALONE MMR BASELINE (no need to re-run full experiment)
# ============================================================
import json, numpy as np, torch, faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, T5ForConditionalGeneration
from collections import Counter

# ---------- 1. Load data and models (fast) ----------
print("Loading corpus and metadata...")
with open("corpus_meta.json", "r") as f:
    saved = json.load(f)
corpus = saved["corpus"]
meta = saved["meta"]
doc_ids = saved["doc_ids"]
embeddings = np.load("embeddings.npy")
index = faiss.IndexFlatL2(1024)
index.add(embeddings)
print(f"Loaded {len(corpus)} comments, FAISS index ready")

print("Loading embedding model...")
embed_model = SentenceTransformer("BAAI/bge-large-en-v1.5")

print("Loading Mistral-7B (4-bit)...")
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
gen_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
gen_model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2", quantization_config=bnb_config, device_map="auto")
gen_model.eval()
EVIDENCE_PROMPT = "You are a competitor intelligence analyst. Answer based strictly on the provided comments. If evidence insufficient, say so."

def hf_generate(system_prompt, user_prompt, max_new_tokens=300):
    prompt = f"[INST] {system_prompt}\n\n{user_prompt} [/INST]"
    inputs = gen_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        out = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=gen_tokenizer.eos_token_id)
    return gen_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("Loading Flan-T5-large judge...")
judge_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
judge_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large", torch_dtype=torch.float16, device_map="auto")
judge_model.eval()

def judge_sentence(sentence, context_texts):
    ctx = " ".join(context_texts) if context_texts else "No evidence provided."
    prompt = f"premise: {ctx} hypothesis: {sentence} Does the premise entail the hypothesis? yes or no"
    inputs = judge_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(judge_model.device)
    with torch.no_grad():
        out = judge_model.generate(**inputs, max_new_tokens=5)
    ans = judge_tokenizer.decode(out[0], skip_special_tokens=True).strip().lower()
    return ans.startswith("no")

def compute_HR(answer, context_texts):
    sents = [s.strip() for s in answer.split(".") if len(s.strip())>10]
    if not sents: return 0.0
    return sum(1 for s in sents if judge_sentence(s, context_texts)) / len(sents)

# ---------- 2. Define QUERIES and RELEVANT (same as in your paper) ----------
# (I will include the full lists here; you already have them in the earlier message)
# For brevity, I assume you have them in your notebook. If not, paste them here.
# I will use the ones from your previous message.

# QUERIES = [ ... ]  # paste your 99 queries from earlier
# RELEVANT = { ... } # paste your relevance dictionary

QUERIES = [
    ("Notion","feature","What AI writing and note-taking features does Notion offer?"),
    ("Notion","feature","Does Notion support collaborative writing or team editing?"),
    ("Notion","feature","What templates and database features does Notion provide?"),
    ("Notion","comparison","How does Notion compare to other productivity tools?"),
    ("Notion","comparison","How does Notion compare to Evernote or Obsidian?"),
    ("Notion","comparison","Is Notion better than traditional note-taking apps?"),
    ("Notion","market","What is the overall user reception of Notion?"),
    ("Notion","market","What do users say about Notion's pricing and value?"),
    ("Notion","market","What are the most common complaints about Notion?"),
    ("Writesonic AI Writer","feature","What AI content generation features does Writesonic offer?"),
    ("Writesonic AI Writer","feature","Does Writesonic support long-form content generation?"),
    ("Writesonic AI Writer","feature","What templates does Writesonic provide for marketers?"),
    ("Writesonic AI Writer","comparison","How does Writesonic compare to Jasper or Copy.ai?"),
    ("Writesonic AI Writer","comparison","Is Writesonic better than other AI writing tools for SEO?"),
    ("Writesonic AI Writer","comparison","How does Writesonic's output quality compare to competitors?"),
    ("Writesonic AI Writer","market","What do users say about Writesonic overall?"),
    ("Writesonic AI Writer","market","What are users' biggest frustrations with Writesonic?"),
    ("Writesonic AI Writer","market","Is Writesonic considered good value for money?"),
    ("Copysmith","feature","What copywriting features does Copysmith provide?"),
    ("Copysmith","feature","Does Copysmith support eCommerce or product description writing?"),
    ("Copysmith","feature","What integrations does Copysmith offer?"),
    ("Copysmith","comparison","How does Copysmith compare to other AI writing tools?"),
    ("Copysmith","comparison","How does Copysmith compare to Jasper for marketing copy?"),
    ("Copysmith","comparison","Is Copysmith better than Copy.ai for eCommerce?"),
    ("Copysmith","market","What is the general user sentiment toward Copysmith?"),
    ("Copysmith","market","What do users say about Copysmith's output quality?"),
    ("Copysmith","market","What are the main limitations users report for Copysmith?"),
    ("compose.ai","feature","What autocomplete features does compose.ai offer?"),
    ("compose.ai","feature","Does compose.ai work across different browsers or platforms?"),
    ("compose.ai","feature","What AI personalization features does compose.ai have?"),
    ("compose.ai","comparison","How does compose.ai compare to other writing assistants?"),
    ("compose.ai","comparison","How does compose.ai compare to Grammarly for autocomplete?"),
    ("compose.ai","comparison","Is compose.ai better than other Chrome extensions for writing?"),
    ("compose.ai","market","What do users think about compose.ai overall?"),
    ("compose.ai","market","What are users' main complaints about compose.ai?"),
    ("compose.ai","market","Is compose.ai considered useful for professional writing?"),
    ("Jasper","feature","What AI writing features does Jasper provide?"),
    ("Jasper","feature","Does Jasper support brand voice customization?"),
    ("Jasper","feature","What long-form writing capabilities does Jasper have?"),
    ("Jasper","comparison","How does Jasper compare to Copy.ai or Writesonic?"),
    ("Jasper","comparison","Is Jasper worth the higher price compared to alternatives?"),
    ("Jasper","comparison","How does Jasper's quality compare to cheaper AI writing tools?"),
    ("Jasper","market","What is the overall user sentiment toward Jasper?"),
    ("Jasper","market","What do users say about Jasper's pricing?"),
    ("Jasper","market","What are the most common Jasper user complaints?"),
    ("Grammarly","feature","What grammar and writing features does Grammarly offer?"),
    ("Grammarly","feature","Does Grammarly support tone detection and style suggestions?"),
    ("Grammarly","feature","What plagiarism detection features does Grammarly have?"),
    ("Grammarly","comparison","How does Grammarly compare to other grammar tools?"),
    ("Grammarly","comparison","How does Grammarly compare to ProWritingAid or Hemingway?"),
    ("Grammarly","comparison","Is Grammarly Premium worth it compared to free alternatives?"),
    ("Grammarly","market","What do users say about Grammarly overall?"),
    ("Grammarly","market","What are users' biggest complaints about Grammarly?"),
    ("Grammarly","market","Do users find Grammarly useful for professional writing?"),
    ("Copy.ai","feature","What content generation features does Copy.ai have?"),
    ("Copy.ai","feature","Does Copy.ai support social media content creation?"),
    ("Copy.ai","feature","What workflow automation features does Copy.ai offer?"),
    ("Copy.ai","comparison","How does Copy.ai differ from Jasper or Writesonic?"),
    ("Copy.ai","comparison","Is Copy.ai better than Jasper for marketing teams?"),
    ("Copy.ai","comparison","How does Copy.ai's free plan compare to competitors?"),
    ("Copy.ai","market","Is Copy.ai effective for marketing teams?"),
    ("Copy.ai","market","What do users say about Copy.ai's ease of use?"),
    ("Copy.ai","market","What are the main limitations of Copy.ai reported by users?"),
    ("Peppertype.ai","feature","What AI writing features does Peppertype.ai offer?"),
    ("Peppertype.ai","feature","Does Peppertype.ai support multiple content formats?"),
    ("Peppertype.ai","feature","What makes Peppertype.ai unique among AI writing tools?"),
    ("Peppertype.ai","comparison","How does Peppertype.ai compare to Copy.ai or Jasper?"),
    ("Peppertype.ai","comparison","Is Peppertype.ai a good alternative to more expensive tools?"),
    ("Peppertype.ai","comparison","How does Peppertype.ai's output quality compare to Jasper?"),
    ("Peppertype.ai","market","What do users think about Peppertype.ai overall?"),
    ("Peppertype.ai","market","What are users' main concerns about Peppertype.ai?"),
    ("Peppertype.ai","market","Is Peppertype.ai considered good value for money?"),
    ("Text Blaze","feature","What text expansion features does Text Blaze offer?"),
    ("Text Blaze","feature","Does Text Blaze support dynamic templates or variables?"),
    ("Text Blaze","feature","What automation features does Text Blaze provide?"),
    ("Text Blaze","comparison","How does Text Blaze compare to other productivity tools?"),
    ("Text Blaze","comparison","How does Text Blaze compare to TextExpander or PhraseExpress?"),
    ("Text Blaze","comparison","Is Text Blaze better than other snippet tools?"),
    ("Text Blaze","market","What is the overall reception of Text Blaze?"),
    ("Text Blaze","market","What do users say about Text Blaze's ease of use?"),
    ("Text Blaze","market","What limitations do users report for Text Blaze?"),
    ("AISEO","feature","What SEO writing features does AISEO provide?"),
    ("AISEO","feature","Does AISEO support keyword optimization and content scoring?"),
    ("AISEO","feature","What AI paraphrasing features does AISEO offer?"),
    ("AISEO","comparison","How does AISEO compare to other SEO tools?"),
    ("AISEO","comparison","How does AISEO compare to Surfer SEO or Frase?"),
    ("AISEO","comparison","Is AISEO a good alternative to more expensive SEO writing tools?"),
    ("AISEO","market","What do users say about AISEO overall?"),
    ("AISEO","market","What are users' main complaints about AISEO?"),
    ("AISEO","market","Is AISEO considered effective for improving search rankings?"),
    ("Hemingway Editor","feature","What editing features does Hemingway Editor have?"),
    ("Hemingway Editor","feature","Does Hemingway Editor support readability scoring?"),
    ("Hemingway Editor","feature","What writing style improvements does Hemingway Editor suggest?"),
    ("Hemingway Editor","comparison","How does Hemingway Editor compare to Grammarly?"),
    ("Hemingway Editor","comparison","Is Hemingway Editor better than Grammarly for style editing?"),
    ("Hemingway Editor","comparison","How does Hemingway Editor compare to ProWritingAid?"),
    ("Hemingway Editor","market","What is the overall sentiment toward Hemingway Editor?"),
    ("Hemingway Editor","market","What do users say about Hemingway Editor's simplicity?"),
    ("Hemingway Editor","market","What are the main limitations of Hemingway Editor?"),
]
print(f"Loaded {len(QUERIES)} queries")

RELEVANT = {
    ("Notion","What AI writing and note-taking features does Notion offer?"): ["P001_C44","P001_C46","P001_C47","P001_C50","P001_C51","P001_C52","P001_C55","P001_C57","P001_C58","P001_C97","P001_C92","P001_C96","P001_C3","P001_C17","P001_C77","P001_C71"],
    ("Notion","Does Notion support collaborative writing or team editing?"): ["P001_C3","P001_C17","P001_C44","P001_C52","P001_C55","P001_C57","P001_C71","P001_C77"],
    ("Notion","What templates and database features does Notion provide?"): ["P001_C46","P001_C47","P001_C50","P001_C51","P001_C58","P001_C92","P001_C96","P001_C97"],
    ("Notion","How does Notion compare to other productivity tools?"): ["P001_C7","P001_C13","P001_C14","P001_C83","P001_C36","P001_C27","P001_C17"],
    ("Notion","How does Notion compare to Evernote or Obsidian?"): ["P001_C7","P001_C13","P001_C14","P001_C27","P001_C36"],
    ("Notion","Is Notion better than traditional note-taking apps?"): ["P001_C7","P001_C13","P001_C14","P001_C83","P001_C36"],
    ("Notion","What is the overall user reception of Notion?"): ["P001_C6","P001_C8","P001_C11","P001_C16","P001_C37","P001_C33","P001_C66","P001_C72","P001_C95","P001_C79","P001_C59","P001_C69"],
    ("Notion","What do users say about Notion's pricing and value?"): ["P001_C6","P001_C8","P001_C16","P001_C33","P001_C37","P001_C59","P001_C66"],
    ("Notion","What are the most common complaints about Notion?"): ["P001_C11","P001_C16","P001_C33","P001_C59","P001_C69","P001_C72","P001_C79"],
    ("Writesonic AI Writer","What AI content generation features does Writesonic offer?"): ["P002_C3","P002_C5","P002_C12","P002_C21","P002_C24","P002_C27","P002_C28","P002_C31","P002_C33","P002_C35","P002_C42"],
    ("Writesonic AI Writer","Does Writesonic support long-form content generation?"): ["P002_C3","P002_C5","P002_C12","P002_C21","P002_C28","P002_C31","P002_C33","P002_C42"],
    ("Writesonic AI Writer","What templates does Writesonic provide for marketers?"): ["P002_C24","P002_C27","P002_C28","P002_C31","P002_C35"],
    ("Writesonic AI Writer","How does Writesonic compare to Jasper or Copy.ai?"): ["P002_C34","P002_C4","P002_C30","P002_C41"],
    ("Writesonic AI Writer","Is Writesonic better than other AI writing tools for SEO?"): ["P002_C4","P002_C30","P002_C34","P002_C41"],
    ("Writesonic AI Writer","How does Writesonic's output quality compare to competitors?"): ["P002_C4","P002_C30","P002_C34","P002_C41"],
    ("Writesonic AI Writer","What do users say about Writesonic overall?"): ["P002_C1","P002_C2","P002_C7","P002_C10","P002_C16","P002_C22","P002_C36","P002_C37","P002_C39","P002_C43"],
    ("Writesonic AI Writer","What are users' biggest frustrations with Writesonic?"): ["P002_C2","P002_C7","P002_C16","P002_C36","P002_C39"],
    ("Writesonic AI Writer","Is Writesonic considered good value for money?"): ["P002_C1","P002_C10","P002_C22","P002_C37","P002_C43"],
    ("Copysmith","What copywriting features does Copysmith provide?"): ["P003_C4","P003_C7","P003_C8","P003_C14","P003_C29","P003_C31"],
    ("Copysmith","Does Copysmith support eCommerce or product description writing?"): ["P003_C4","P003_C7","P003_C8","P003_C14","P003_C29"],
    ("Copysmith","What integrations does Copysmith offer?"): ["P003_C7","P003_C8","P003_C14","P003_C31"],
    ("Copysmith","How does Copysmith compare to other AI writing tools?"): ["P003_C10","P003_C15","P003_C17","P003_C22"],
    ("Copysmith","How does Copysmith compare to Jasper for marketing copy?"): ["P003_C10","P003_C15","P003_C17","P003_C22"],
    ("Copysmith","Is Copysmith better than Copy.ai for eCommerce?"): ["P003_C10","P003_C15","P003_C17"],
    ("Copysmith","What is the general user sentiment toward Copysmith?"): ["P003_C5","P003_C9","P003_C15","P003_C20","P003_C21","P003_C22","P003_C23","P003_C24","P003_C26","P003_C27"],
    ("Copysmith","What do users say about Copysmith's output quality?"): ["P003_C5","P003_C9","P003_C20","P003_C21","P003_C23"],
    ("Copysmith","What are the main limitations users report for Copysmith?"): ["P003_C15","P003_C21","P003_C24","P003_C26","P003_C27"],
    ("compose.ai","What autocomplete features does compose.ai offer?"): ["P004_C2","P004_C3","P004_C4","P004_C5","P004_C9","P004_C15","P004_C22","P004_C24"],
    ("compose.ai","Does compose.ai work across different browsers or platforms?"): ["P004_C2","P004_C3","P004_C4","P004_C9","P004_C15"],
    ("compose.ai","What AI personalization features does compose.ai have?"): ["P004_C4","P004_C5","P004_C22","P004_C24"],
    ("compose.ai","How does compose.ai compare to other writing assistants?"): ["P004_C9","P004_C20","P004_C6"],
    ("compose.ai","How does compose.ai compare to Grammarly for autocomplete?"): ["P004_C6","P004_C9","P004_C20"],
    ("compose.ai","Is compose.ai better than other Chrome extensions for writing?"): ["P004_C6","P004_C9","P004_C20"],
    ("compose.ai","What do users think about compose.ai overall?"): ["P004_C2","P004_C7","P004_C8","P004_C10","P004_C17","P004_C20","P004_C24"],
    ("compose.ai","What are users' main complaints about compose.ai?"): ["P004_C7","P004_C8","P004_C10","P004_C17"],
    ("compose.ai","Is compose.ai considered useful for professional writing?"): ["P004_C2","P004_C10","P004_C20","P004_C24"],
    ("Jasper","What AI writing features does Jasper provide?"): ["P005_C1","P005_C4","P005_C5","P005_C6","P005_C7","P005_C15","P005_C17","P005_C18","P005_C19","P005_C20","P005_C38"],
    ("Jasper","Does Jasper support brand voice customization?"): ["P005_C4","P005_C5","P005_C6","P005_C15","P005_C17","P005_C18","P005_C19","P005_C20"],
    ("Jasper","What long-form writing capabilities does Jasper have?"): ["P005_C1","P005_C4","P005_C7","P005_C15","P005_C38"],
    ("Jasper","How does Jasper compare to Copy.ai or Writesonic?"): ["P005_C11","P005_C13","P005_C35","P005_C10"],
    ("Jasper","Is Jasper worth the higher price compared to alternatives?"): ["P005_C10","P005_C11","P005_C13","P005_C35"],
    ("Jasper","How does Jasper's quality compare to cheaper AI writing tools?"): ["P005_C10","P005_C11","P005_C13","P005_C35"],
    ("Jasper","What is the overall user sentiment toward Jasper?"): ["P005_C2","P005_C3","P005_C9","P005_C14","P005_C29","P005_C33","P005_C36","P005_C10"],
    ("Jasper","What do users say about Jasper's pricing?"): ["P005_C2","P005_C3","P005_C9","P005_C29","P005_C33"],
    ("Jasper","What are the most common Jasper user complaints?"): ["P005_C3","P005_C9","P005_C14","P005_C29","P005_C36"],
    ("Grammarly","What grammar and writing features does Grammarly offer?"): ["P006_C1","P006_C9","P006_C25","P006_C35","P006_C41","P006_C43","P006_C44","P006_C47","P006_C51","P006_C52"],
    ("Grammarly","Does Grammarly support tone detection and style suggestions?"): ["P006_C9","P006_C25","P006_C35","P006_C41","P006_C43","P006_C44","P006_C47"],
    ("Grammarly","What plagiarism detection features does Grammarly have?"): ["P006_C1","P006_C25","P006_C41","P006_C51","P006_C52"],
    ("Grammarly","How does Grammarly compare to other grammar tools?"): ["P006_C11","P006_C14","P006_C29","P006_C47","P006_C49","P006_C50"],
    ("Grammarly","How does Grammarly compare to ProWritingAid or Hemingway?"): ["P006_C11","P006_C14","P006_C29","P006_C49","P006_C50"],
    ("Grammarly","Is Grammarly Premium worth it compared to free alternatives?"): ["P006_C11","P006_C14","P006_C47","P006_C49","P006_C50"],
    ("Grammarly","What do users say about Grammarly overall?"): ["P006_C1","P006_C4","P006_C21","P006_C24","P006_C26","P006_C37","P006_C38","P006_C46","P006_C48"],
    ("Grammarly","What are users' biggest complaints about Grammarly?"): ["P006_C4","P006_C21","P006_C26","P006_C37","P006_C38"],
    ("Grammarly","Do users find Grammarly useful for professional writing?"): ["P006_C1","P006_C24","P006_C46","P006_C48"],
    ("Copy.ai","What content generation features does Copy.ai have?"): ["P007_C3","P007_C19","P007_C28","P007_C34","P007_C36","P007_C45","P007_C47","P007_C52","P007_C58","P007_C66"],
    ("Copy.ai","Does Copy.ai support social media content creation?"): ["P007_C3","P007_C19","P007_C28","P007_C34","P007_C45","P007_C47","P007_C52"],
    ("Copy.ai","What workflow automation features does Copy.ai offer?"): ["P007_C28","P007_C34","P007_C36","P007_C47","P007_C58","P007_C66"],
    ("Copy.ai","How does Copy.ai differ from Jasper or Writesonic?"): ["P007_C40","P007_C41","P007_C49","P007_C50","P007_C61"],
    ("Copy.ai","Is Copy.ai better than Jasper for marketing teams?"): ["P007_C40","P007_C41","P007_C49","P007_C50"],
    ("Copy.ai","How does Copy.ai's free plan compare to competitors?"): ["P007_C40","P007_C49","P007_C50","P007_C61"],
    ("Copy.ai","Is Copy.ai effective for marketing teams?"): ["P007_C1","P007_C5","P007_C6","P007_C8","P007_C10","P007_C18","P007_C21","P007_C46","P007_C53","P007_C56"],
    ("Copy.ai","What do users say about Copy.ai's ease of use?"): ["P007_C1","P007_C5","P007_C6","P007_C8","P007_C18","P007_C21","P007_C46"],
    ("Copy.ai","What are the main limitations of Copy.ai reported by users?"): ["P007_C10","P007_C53","P007_C56","P007_C61"],
    ("Peppertype.ai","What AI writing features does Peppertype.ai offer?"): ["P008_C1","P008_C2","P008_C3","P008_C8","P008_C9","P008_C15","P008_C16","P008_C17","P008_C18","P008_C19","P008_C31","P008_C32"],
    ("Peppertype.ai","Does Peppertype.ai support multiple content formats?"): ["P008_C1","P008_C2","P008_C8","P008_C15","P008_C16","P008_C17","P008_C18","P008_C19"],
    ("Peppertype.ai","What makes Peppertype.ai unique among AI writing tools?"): ["P008_C3","P008_C9","P008_C31","P008_C32"],
    ("Peppertype.ai","How does Peppertype.ai compare to Copy.ai or Jasper?"): ["P008_C5","P008_C11","P008_C12","P008_C24","P008_C25","P008_C50"],
    ("Peppertype.ai","Is Peppertype.ai a good alternative to more expensive tools?"): ["P008_C5","P008_C11","P008_C24","P008_C25","P008_C50"],
    ("Peppertype.ai","How does Peppertype.ai's output quality compare to Jasper?"): ["P008_C5","P008_C12","P008_C24","P008_C25"],
    ("Peppertype.ai","What do users think about Peppertype.ai overall?"): ["P008_C1","P008_C2","P008_C3","P008_C6","P008_C22","P008_C34","P008_C35","P008_C39","P008_C40","P008_C45"],
    ("Peppertype.ai","What are users' main concerns about Peppertype.ai?"): ["P008_C6","P008_C22","P008_C34","P008_C39","P008_C40"],
    ("Peppertype.ai","Is Peppertype.ai considered good value for money?"): ["P008_C1","P008_C2","P008_C35","P008_C45"],
    ("Text Blaze","What text expansion features does Text Blaze offer?"): ["P009_C1","P009_C2","P009_C3","P009_C7","P009_C8","P009_C13","P009_C15","P009_C16","P009_C32"],
    ("Text Blaze","Does Text Blaze support dynamic templates or variables?"): ["P009_C1","P009_C2","P009_C7","P009_C8","P009_C13","P009_C15","P009_C32"],
    ("Text Blaze","What automation features does Text Blaze provide?"): ["P009_C3","P009_C7","P009_C8","P009_C15","P009_C16"],
    ("Text Blaze","How does Text Blaze compare to other productivity tools?"): ["P009_C10","P009_C20","P009_C29","P009_C39"],
    ("Text Blaze","How does Text Blaze compare to TextExpander or PhraseExpress?"): ["P009_C10","P009_C20","P009_C29","P009_C39"],
    ("Text Blaze","Is Text Blaze better than other snippet tools?"): ["P009_C10","P009_C20","P009_C29"],
    ("Text Blaze","What is the overall reception of Text Blaze?"): ["P009_C1","P009_C13","P009_C16","P009_C22","P009_C25","P009_C18","P009_C31","P009_C34"],
    ("Text Blaze","What do users say about Text Blaze's ease of use?"): ["P009_C1","P009_C13","P009_C16","P009_C22","P009_C25"],
    ("Text Blaze","What limitations do users report for Text Blaze?"): ["P009_C18","P009_C31","P009_C34"],
    ("AISEO","What SEO writing features does AISEO provide?"): ["P010_C2","P010_C17","P010_C18","P010_C23","P010_C24","P010_C25","P010_C28","P010_C31","P010_C32","P010_C16A","P010_C22"],
    ("AISEO","Does AISEO support keyword optimization and content scoring?"): ["P010_C17","P010_C18","P010_C23","P010_C24","P010_C25","P010_C28","P010_C31","P010_C32"],
    ("AISEO","What AI paraphrasing features does AISEO offer?"): ["P010_C2","P010_C16A","P010_C22","P010_C23","P010_C24"],
    ("AISEO","How does AISEO compare to other SEO tools?"): ["P010_C5","P010_C8","P010_C15B","P010_C16B","P010_C33"],
    ("AISEO","How does AISEO compare to Surfer SEO or Frase?"): ["P010_C5","P010_C8","P010_C15B","P010_C16B"],
    ("AISEO","Is AISEO a good alternative to more expensive SEO writing tools?"): ["P010_C5","P010_C8","P010_C33"],
    ("AISEO","What do users say about AISEO overall?"): ["P010_C1","P010_C6","P010_C9","P010_C10","P010_C16A","P010_C19","P010_C21","P010_C26","P010_C29","P010_C33"],
    ("AISEO","What are users' main complaints about AISEO?"): ["P010_C6","P010_C9","P010_C19","P010_C21","P010_C26"],
    ("AISEO","Is AISEO considered effective for improving search rankings?"): ["P010_C1","P010_C10","P010_C29","P010_C33"],
    ("Hemingway Editor","What editing features does Hemingway Editor have?"): ["P011_C5","P011_C6","P011_C15","P011_C16","P011_C19","P011_C20","P011_C22","P011_C24","P011_C35","P011_C40","P011_C43"],
    ("Hemingway Editor","Does Hemingway Editor support readability scoring?"): ["P011_C5","P011_C6","P011_C15","P011_C16","P011_C19","P011_C20","P011_C22","P011_C24"],
    ("Hemingway Editor","What writing style improvements does Hemingway Editor suggest?"): ["P011_C15","P011_C19","P011_C20","P011_C35","P011_C40","P011_C43"],
    ("Hemingway Editor","How does Hemingway Editor compare to Grammarly?"): ["P011_C7","P011_C11","P011_C26","P011_C39"],
    ("Hemingway Editor","Is Hemingway Editor better than Grammarly for style editing?"): ["P011_C7","P011_C11","P011_C26","P011_C39"],
    ("Hemingway Editor","How does Hemingway Editor compare to ProWritingAid?"): ["P011_C7","P011_C11","P011_C26"],
    ("Hemingway Editor","What is the overall sentiment toward Hemingway Editor?"): ["P011_C3","P011_C9","P011_C12","P011_C15","P011_C18","P011_C28","P011_C33","P011_C34","P011_C44","P011_C45"],
    ("Hemingway Editor","What do users say about Hemingway Editor's simplicity?"): ["P011_C3","P011_C9","P011_C12","P011_C28","P011_C33"],
    ("Hemingway Editor","What are the main limitations of Hemingway Editor?"): ["P011_C15","P011_C18","P011_C34","P011_C44","P011_C45"],
}
print(f"Loaded {len(RELEVANT)} relevance maps")

id2sent = {m["doc_id"]: m["sentiment"] for m in meta}

# ---------- 3. MMR functions ----------
def mmr_retrieval(query, pool_size=50, k=5, lambda_=0.5):
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    D, I = index.search(q_emb, pool_size)
    candidates = []
    for idx in range(len(I[0])):
        doc_id = doc_ids[I[0][idx]]
        text = corpus[I[0][idx]]
        m = meta[I[0][idx]]
        sim_q = 1.0 / (1.0 + D[0][idx])
        emb = embeddings[I[0][idx]]
        candidates.append({"doc_id": doc_id, "text": text, "meta": m, "sim_q": sim_q, "embedding": emb})
    selected, selected_idx = [], []
    for _ in range(min(k, len(candidates))):
        best_score, best_i = -1e9, -1
        for i, cand in enumerate(candidates):
            if i in selected_idx: continue
            rel = cand["sim_q"]
            max_sim = max(np.dot(cand["embedding"], sel["embedding"]) for sel in selected) if selected else 0.0
            mmr_score = lambda_ * rel - (1 - lambda_) * max_sim
            if mmr_score > best_score:
                best_score, best_i = mmr_score, i
        selected.append(candidates[best_i])
        selected_idx.append(best_i)
    return [s["doc_id"] for s in selected], [s["text"] for s in selected], [s["meta"] for s in selected]

def run_MMR(query, k=5, pool=50, lambda_=0.5):
    dids, txts, mts = mmr_retrieval(query, pool_size=pool, k=k, lambda_=lambda_)
    ctx = "\n".join(f"[{m['subproduct']}] {t}" for t,m in zip(txts,mts))
    ans = hf_generate(EVIDENCE_PROMPT, f"Evidence:\n{ctx}\n\nQuery: {query}")
    return {"answer": ans, "doc_ids": dids, "texts": txts}

# ---------- 4. Run MMR ----------
print("\nRunning MMR on 99 queries...")
mmr_results = []
for i, (prod, qtype, q) in enumerate(QUERIES):
    print(f"  [{i+1}/99] {prod} – {qtype}")
    mmr_results.append(run_MMR(q, k=5, pool=50, lambda_=0.5))

# ---------- 5. Compute metrics ----------
mmr_hr = [compute_HR(r["answer"], r["texts"]) for r in mmr_results]
mmr_ids = [r["doc_ids"] for r in mmr_results]

def sbs_from_ids(hr_list, id_lists):
    pos, neg = [], []
    for hr, ids in zip(hr_list, id_lists):
        if not ids: continue
        dom = Counter(id2sent.get(d,"neutral") for d in ids).most_common(1)[0][0]
        if dom == "positive": pos.append(hr)
        elif dom == "negative": neg.append(hr)
    return (np.mean(pos) if pos else 0.0) - (np.mean(neg) if neg else 0.0)

mmr_sbs = sbs_from_ids(mmr_hr, mmr_ids)

def p_at_k(ret, rel, k):
    return sum(1 for d in ret[:k] if d in rel) / k if k>0 else 0.0
def r_at_k(ret, rel, k):
    return sum(1 for d in ret[:k] if d in rel) / max(len(rel),1)

mmr_p5, mmr_r5 = [], []
for (prod,_,q), res in zip(QUERIES, mmr_results):
    rel = RELEVANT.get((prod,q), [])
    ret = res["doc_ids"]
    mmr_p5.append(p_at_k(ret, rel, 5))
    mmr_r5.append(r_at_k(ret, rel, 5))

print("\n" + "="*60)
print("MMR BASELINE RESULTS (λ=0.5, k=5, POOL=50)")
print("="*60)
print(f"  HR (macro) : {np.mean(mmr_hr):.3f}")
print(f"  SBS        : {mmr_sbs:.3f}")
print(f"  P@5        : {np.mean(mmr_p5):.3f}")
print(f"  R@5        : {np.mean(mmr_r5):.3f}")
print("="*60)

Loading corpus and metadata...
Loaded 536 comments, FAISS index ready
Loading embedding model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading Mistral-7B (4-bit)...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded 99 queries
Loaded 99 relevance maps

Running MMR on 99 queries...
  [1/99] Notion – feature
  [2/99] Notion – feature
  [3/99] Notion – feature
  [4/99] Notion – comparison
  [5/99] Notion – comparison
  [6/99] Notion – comparison
  [7/99] Notion – market
  [8/99] Notion – market
  [9/99] Notion – market
  [10/99] Writesonic AI Writer – feature
  [11/99] Writesonic AI Writer – feature
  [12/99] Writesonic AI Writer – feature
  [13/99] Writesonic AI Writer – comparison
  [14/99] Writesonic AI Writer – comparison
  [15/99] Writesonic AI Writer – comparison
  [16/99] Writesonic AI Writer – market
  [17/99] Writesonic AI Writer – market
  [18/99] Writesonic AI Writer – market
  [19/99] Copysmith – feature
  [20/99] Copysmith – feature
  [21/99] Copysmith – feature
  [22/99] Copysmith – comparison
  [23/99] Copysmith – comparison
  [24/99] Copysmith – comparison
  [25/99] Copysmith – market
  [26/99] Copysmith – market
  [27/99] Copysmith – market
  [28/99] compose.ai – feature
  [29